In [2]:
import tensorflow as tf
import tensorflow_model_optimization as tfmot
from codecarbon import EmissionsTracker
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import pandas as pd
import numpy as np
import time
import os

# Sonuç listesi
results = []

# Sonuçları kaydetme fonksiyonu
def log_result(model_name, technique, y_true, y_pred, train_time, emissions_data, model_size_mb=None):
    result = {
        'Model': model_name,
        'Teknik': technique,
        'F1': round(f1_score(y_true, y_pred, average='weighted'), 4),
        'Precision': round(precision_score(y_true, y_pred, average='weighted'), 4),
        'Recall': round(recall_score(y_true, y_pred, average='weighted'), 4),
        'Accuracy': round(accuracy_score(y_true, y_pred), 4),
        'Eğitim Süresi (sn)': round(train_time, 2),
        'Enerji (kWh)': round(emissions_data.energy_consumed, 6) if emissions_data else None,
        'CO2 (kg)': round(emissions_data.emissions, 6) if emissions_data else None,
        'Model Boyutu (MB)': round(model_size_mb, 2) if model_size_mb else None
    }
    results.append(result)
    print(f"\n{'='*50}")
    for k, v in result.items():
        print(f"  {k}: {v}")
    print(f"{'='*50}")
    return result

print("TensorFlow versiyonu:", tf.__version__)
print("GPU kullanılabilir mi:", len(tf.config.list_physical_devices('GPU')) > 0)


TensorFlow versiyonu: 2.21.0
GPU kullanılabilir mi: False


In [3]:
# CIFAR-10 yükleme 
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

y_train = y_train.flatten()
y_test = y_test.flatten()

# Normalize
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

IMG_SIZE = 64

# Bellek dostu resize: 10K'lık parçalar halinde
def batch_resize(images, size, batch_size=10000):
    resized = []
    for i in range(0, len(images), batch_size):
        batch = tf.image.resize(images[i:i+batch_size], [size, size])
        resized.append(batch.numpy())
    return np.concatenate(resized, axis=0)

print("Train resize ediliyor...")
x_train = batch_resize(x_train, IMG_SIZE)
print("Test resize ediliyor...")
x_test = batch_resize(x_test, IMG_SIZE)

# Train'den %10 validation ayır
from sklearn.model_selection import train_test_split
x_train_final, x_val, y_train_final, y_val = train_test_split(
    x_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

print(f"Train: {x_train_final.shape}")
print(f"Validation: {x_val.shape}")
print(f"Test: {x_test.shape}")

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']
print(f"Sınıf sayısı: {len(class_names)}")

Train resize ediliyor...
Test resize ediliyor...
Train: (45000, 64, 64, 3)
Validation: (5000, 64, 64, 3)
Test: (10000, 64, 64, 3)
Sınıf sayısı: 10


In [4]:
# MobileNetV2 - Transfer Learning
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Pretrained ağırlıkları dondur

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()
print(f"\nToplam parametre: {model.count_params():,}")



9406464/9406464 [==============================] - 3s 0us/step

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 mobilenetv2_1.00_224 (Func  (None, 2, 2, 1280)        2257984   
 tional)                                                         
                                                                 
 global_average_pooling2d (  (None, 1280)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 dropout (Dropout)           (None, 1280)              0         
                                                                 
 dense (Dense)               (None, 10)                12810     
                                                                 
Total params: 2270794 (8.66 MB)
Trainable params: 12810 (50.04 KB)
Non-trainable params: 2257984 (8.61 MB)
_______________

In [6]:
# CodeCarbon GPU spam'ini önle
import os
os.environ["CODECARBON_GPU_IDS"] = ""  # GPU izlemeyi devre dışı bırak

from codecarbon import OfflineEmissionsTracker

BATCH_SIZE = 32
EPOCHS = 10

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

# OfflineEmissionsTracker — internet bağlantısı gerektirmez, GPU izlemez
tracker = OfflineEmissionsTracker(
    project_name="mobilenet_baseline",
    country_iso_code="TUR",
    log_level="error"  # sadece gerçek hataları göster
)

tracker.start()
start_time = time.time()

history_baseline = model.fit(
    x_train_final, y_train_final,
    validation_data=(x_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)

train_time = time.time() - start_time
emissions_baseline = tracker.stop()

# Test tahminleri
y_pred = model.predict(x_test, verbose=0)
y_pred_classes = np.argmax(y_pred, axis=1)

[codecarbon WARNING @ 21:07:50] Multiple instances of codecarbon are allowed to run at the same time.


Epoch 1/10


1407/1407 [==============================] - 157s 103ms/step - loss: 1.2715 - accuracy: 0.5669 - val_loss: 1.0424 - val_accuracy: 0.6472
Epoch 2/10
1407/1407 [==============================] - 122s 86ms/step - loss: 1.0794 - accuracy: 0.6336 - val_loss: 1.0204 - val_accuracy: 0.6490
Epoch 3/10
1407/1407 [==============================] - 131s 93ms/step - loss: 1.0567 - accuracy: 0.6394 - val_loss: 1.0215 - val_accuracy: 0.6526
Epoch 4/10
1407/1407 [==============================] - 127s 90ms/step - loss: 1.0422 - accuracy: 0.6452 - val_loss: 1.0067 - val_accuracy: 0.6532
Epoch 5/10
1407/1407 [==============================] - 126s 90ms/step - loss: 1.0346 - accuracy: 0.6464 - val_loss: 1.0219 - val_accuracy: 0.6566
Epoch 6/10
1407/1407 [==============================] - 129s 92ms/step - loss: 1.0364 - accuracy: 0.6459 - val_loss: 1.0259 - val_accuracy: 0.6498
Epoch 7/10
1407/1407 [==============================] - 126s 89ms/step - loss: 1.0337 - accuracy: 0.6437 - val_loss

FileNotFoundError: [Errno 2] No such file or directory: 'results/mobilenet_baseline.keras'

In [9]:
#Modeli kaydet
model.save('../results/mobilenet_baseline.keras')
baseline_size = os.path.getsize('../results/mobilenet_baseline.keras') / (1024 * 1024)

# Sonuçları logla (düzeltilmiş)
result = {
    'Model': 'MobileNetV2',
    'Teknik': 'Baseline (Early Stopping)',
    'F1': round(f1_score(y_test, y_pred_classes, average='weighted'), 4),
    'Precision': round(precision_score(y_test, y_pred_classes, average='weighted'), 4),
    'Recall': round(recall_score(y_test, y_pred_classes, average='weighted'), 4),
    'Accuracy': round(accuracy_score(y_test, y_pred_classes), 4),
    'Eğitim Süresi (sn)': round(train_time, 2),
    'Enerji (kWh)': round(tracker.final_emissions_data.energy_consumed, 6),
    'CO2 (kg)': round(tracker.final_emissions, 6),
    'Model Boyutu (MB)': round(baseline_size, 2)
}
results.append(result)

for k, v in result.items():
    print(f"  {k}: {v}")

  Model: MobileNetV2
  Teknik: Baseline (Early Stopping)
  F1: 0.6522
  Precision: 0.655
  Recall: 0.6547
  Accuracy: 0.6547
  Eğitim Süresi (sn): 1038.72
  Enerji (kWh): 0.008409
  CO2 (kg): 0.003907
  Model Boyutu (MB): 9.27


In [10]:
# log_result fonksiyonunu güncelle
def log_result(model_name, technique, y_true, y_pred, train_time, tracker_obj, model_size_mb=None):
    result = {
        'Model': model_name,
        'Teknik': technique,
        'F1': round(f1_score(y_true, y_pred, average='weighted'), 4),
        'Precision': round(precision_score(y_true, y_pred, average='weighted'), 4),
        'Recall': round(recall_score(y_true, y_pred, average='weighted'), 4),
        'Accuracy': round(accuracy_score(y_true, y_pred), 4),
        'Eğitim Süresi (sn)': round(train_time, 2),
        'Enerji (kWh)': round(tracker_obj.final_emissions_data.energy_consumed, 6) if tracker_obj else None,
        'CO2 (kg)': round(tracker_obj.final_emissions, 6) if tracker_obj else None,
        'Model Boyutu (MB)': round(model_size_mb, 2) if model_size_mb else None
    }
    results.append(result)
    print(f"\n{'='*50}")
    for k, v in result.items():
        print(f"  {k}: {v}")
    print(f"{'='*50}")
    return result

In [11]:
# Post-Training Quantization (FP32 → INT8)
import tensorflow as tf

# TFLite converter ile quantize et
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Quantization için kalibrasyon verisi (representative dataset)
def representative_dataset():
    for i in range(100):
        yield [x_train_final[i:i+1].astype(np.float32)]

converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_model = converter.convert()

# Quantized modeli kaydet
tflite_path = '../results/mobilenet_quantized.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

quant_size = os.path.getsize(tflite_path) / (1024 * 1024)
print(f"Baseline model boyutu: {baseline_size:.2f} MB")
print(f"Quantized model boyutu: {quant_size:.2f} MB")
print(f"Küçülme oranı: {(1 - quant_size/baseline_size)*100:.1f}%")

# Quantized model ile inference
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Inference süresi ve enerji ölçümü
tracker_quant = OfflineEmissionsTracker(
    project_name="mobilenet_quantized",
    country_iso_code="TUR",
    log_level="error"
)

tracker_quant.start()
start_time = time.time()

y_pred_quant = []
for i in range(len(x_test)):
    input_data = x_test[i:i+1]
    
    # INT8 modeli uint8 input bekliyor
    if input_details[0]['dtype'] == np.uint8:
        input_scale, input_zero_point = input_details[0]['quantization']
        input_data = (input_data / input_scale + input_zero_point).astype(np.uint8)
    
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    y_pred_quant.append(np.argmax(output))

inference_time = time.time() - start_time
tracker_quant.stop()

y_pred_quant = np.array(y_pred_quant)

log_result(
    model_name='MobileNetV2',
    technique='Quantization (INT8)',
    y_true=y_test,
    y_pred=y_pred_quant,
    train_time=inference_time,
    tracker_obj=tracker_quant,
    model_size_mb=quant_size
)

INFO:tensorflow:Assets written to: C:\Users\bahar\AppData\Local\Temp\tmp27rt4qpm\assets


INFO:tensorflow:Assets written to: C:\Users\bahar\AppData\Local\Temp\tmp27rt4qpm\assets
C:\Users\bahar\source\repos\green-ai-project\venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Baseline model boyutu: 9.27 MB
Quantized model boyutu: 2.60 MB
Küçülme oranı: 71.9%


C:\Users\bahar\source\repos\green-ai-project\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



  Model: MobileNetV2
  Teknik: Quantization (INT8)
  F1: 0.6253
  Precision: 0.6336
  Recall: 0.6286
  Accuracy: 0.6286
  Eğitim Süresi (sn): 19.93
  Enerji (kWh): 0.00012
  CO2 (kg): 5.6e-05
  Model Boyutu (MB): 2.6


{'Model': 'MobileNetV2',
 'Teknik': 'Quantization (INT8)',
 'F1': 0.6253,
 'Precision': 0.6336,
 'Recall': 0.6286,
 'Accuracy': 0.6286,
 'Eğitim Süresi (sn)': 19.93,
 'Enerji (kWh)': 0.00012,
 'CO2 (kg)': 5.6e-05,
 'Model Boyutu (MB)': 2.6}

In [12]:
# Pruning - Magnitude-based (%50 sparsity)

# Yeni model oluştur (aynı mimari)
base_model_p = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model_p.trainable = False

model_for_pruning = tf.keras.Sequential([
    base_model_p,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation='softmax')
])

# Baseline ağırlıklarını yükle
model_for_pruning.load_weights('../results/mobilenet_baseline.keras')

# Pruning parametreleri
pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0,
        final_sparsity=0.5,
        begin_step=0,
        end_step=1000
    )
}

# Sadece Dense katmanı prune edilecek
def apply_pruning_to_dense(layer):
    if isinstance(layer, tf.keras.layers.Dense):
        return tfmot.sparsity.keras.prune_low_magnitude(layer, **pruning_params)
    return layer

model_pruned = tf.keras.models.clone_model(
    model_for_pruning,
    clone_function=apply_pruning_to_dense
)

model_pruned.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Pruning callback
callbacks_pruning = [
    tfmot.sparsity.keras.UpdatePruningStep(),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
]

# Eğitim
tracker_prune = OfflineEmissionsTracker(
    project_name="mobilenet_pruning",
    country_iso_code="TUR",
    log_level="error"
)

tracker_prune.start()
start_time = time.time()

history_pruned = model_pruned.fit(
    x_train_final, y_train_final,
    validation_data=(x_val, y_val),
    epochs=5,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_pruning,
    verbose=1
)

train_time_prune = time.time() - start_time
tracker_prune.stop()

# Pruning wrappers'ı kaldır
model_stripped = tfmot.sparsity.keras.strip_pruning(model_pruned)

# Kaydet ve boyut ölç
model_stripped.save('../results/mobilenet_pruned.keras')
pruned_size = os.path.getsize('../results/mobilenet_pruned.keras') / (1024 * 1024)

# Test
y_pred_pruned = np.argmax(model_stripped.predict(x_test, verbose=0), axis=1)

log_result(
    model_name='MobileNetV2',
    technique='Pruning (%50 Sparsity)',
    y_true=y_test,
    y_pred=y_pred_pruned,
    train_time=train_time_prune,
    tracker_obj=tracker_prune,
    model_size_mb=pruned_size
)

Epoch 1/5
1407/1407 [==============================] - 105s 71ms/step - loss: 1.0405 - accuracy: 0.6424 - val_loss: 1.0206 - val_accuracy: 0.6524
Epoch 2/5
1407/1407 [==============================] - 76s 54ms/step - loss: 1.0210 - accuracy: 0.6468 - val_loss: 1.0226 - val_accuracy: 0.6516
Epoch 3/5
1407/1407 [==============================] - 72s 51ms/step - loss: 1.0170 - accuracy: 0.6502 - val_loss: 1.0247 - val_accuracy: 0.6496
Epoch 4/5
1407/1407 [==============================] - 75s 53ms/step - loss: 1.0161 - accuracy: 0.6494 - val_loss: 1.0237 - val_accuracy: 0.6494

  Model: MobileNetV2
  Teknik: Pruning (%50 Sparsity)
  F1: 0.642
  Precision: 0.6434
  Recall: 0.6441
  Accuracy: 0.6441
  Eğitim Süresi (sn): 377.24
  Enerji (kWh): 0.002912
  CO2 (kg): 0.001353
  Model Boyutu (MB): 9.16


{'Model': 'MobileNetV2',
 'Teknik': 'Pruning (%50 Sparsity)',
 'F1': 0.642,
 'Precision': 0.6434,
 'Recall': 0.6441,
 'Accuracy': 0.6441,
 'Eğitim Süresi (sn)': 377.24,
 'Enerji (kWh)': 0.002912,
 'CO2 (kg)': 0.001353,
 'Model Boyutu (MB)': 9.16}

In [13]:
df_results = pd.DataFrame(results)
df_results.to_csv('../results/mobilenet_results.csv', index=False)
print(df_results.to_string(index=False))

      Model                    Teknik     F1  Precision  Recall  Accuracy  Eğitim Süresi (sn)  Enerji (kWh)  CO2 (kg)  Model Boyutu (MB)
MobileNetV2 Baseline (Early Stopping) 0.6522     0.6550  0.6547    0.6547             1038.72      0.008409  0.003907               9.27
MobileNetV2       Quantization (INT8) 0.6253     0.6336  0.6286    0.6286               19.93      0.000120  0.000056               2.60
MobileNetV2    Pruning (%50 Sparsity) 0.6420     0.6434  0.6441    0.6441              377.24      0.002912  0.001353               9.16


In [14]:
# Pruning v2 - Tüm model unfreeze edilmiş

base_model_p2 = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model_p2.trainable = True  # UNFREEZE

model_for_pruning_v2 = tf.keras.Sequential([
    base_model_p2,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation='softmax')
])

# Baseline ağırlıklarını yükle
model_for_pruning_v2.load_weights('../results/mobilenet_baseline.keras')

# Tüm katmanlara pruning uygula
pruning_params_v2 = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0,
        final_sparsity=0.5,
        begin_step=0,
        end_step=2000
    )
}

model_pruned_v2 = tfmot.sparsity.keras.prune_low_magnitude(
    model_for_pruning_v2, **pruning_params_v2
)

model_pruned_v2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),  # düşük lr — fine-tuning
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_pruning_v2 = [
    tfmot.sparsity.keras.UpdatePruningStep(),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
]

tracker_prune_v2 = OfflineEmissionsTracker(
    project_name="mobilenet_pruning_v2",
    country_iso_code="TUR",
    log_level="error"
)

print("Pruning v2 başlıyor (tüm model unfreeze)...")
tracker_prune_v2.start()
start_time = time.time()

history_pruned_v2 = model_pruned_v2.fit(
    x_train_final, y_train_final,
    validation_data=(x_val, y_val),
    epochs=5,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_pruning_v2,
    verbose=1
)

train_time_prune_v2 = time.time() - start_time
tracker_prune_v2.stop()

# Strip pruning
model_stripped_v2 = tfmot.sparsity.keras.strip_pruning(model_pruned_v2)

# Kaydet
model_stripped_v2.save('../results/mobilenet_pruned_v2.keras')
pruned_v2_size = os.path.getsize('../results/mobilenet_pruned_v2.keras') / (1024 * 1024)

# Test
y_pred_pruned_v2 = np.argmax(model_stripped_v2.predict(x_test, verbose=0), axis=1)

log_result(
    model_name='MobileNetV2',
    technique='Pruning v2 (%50 Sparsity - Full Model)',
    y_true=y_test,
    y_pred=y_pred_pruned_v2,
    train_time=train_time_prune_v2,
    tracker_obj=tracker_prune_v2,
    model_size_mb=pruned_v2_size
)

Pruning v2 başlıyor (tüm model unfreeze)...
Epoch 1/5
1407/1407 [==============================] - 364s 224ms/step - loss: 1.2339 - accuracy: 0.6109 - val_loss: 3.5298 - val_accuracy: 0.3000
Epoch 2/5
1407/1407 [==============================] - 293s 208ms/step - loss: 0.7555 - accuracy: 0.7427 - val_loss: 0.6742 - val_accuracy: 0.7784
Epoch 3/5
1407/1407 [==============================] - 295s 210ms/step - loss: 0.5751 - accuracy: 0.8004 - val_loss: 0.5329 - val_accuracy: 0.8116
Epoch 4/5
1407/1407 [==============================] - 293s 208ms/step - loss: 0.4669 - accuracy: 0.8372 - val_loss: 0.5041 - val_accuracy: 0.8340
Epoch 5/5
1407/1407 [==============================] - 292s 207ms/step - loss: 0.3891 - accuracy: 0.8634 - val_loss: 0.4533 - val_accuracy: 0.8516

  Model: MobileNetV2
  Teknik: Pruning v2 (%50 Sparsity - Full Model)
  F1: 0.8405
  Precision: 0.8442
  Recall: 0.8406
  Accuracy: 0.8406
  Eğitim Süresi (sn): 1584.48
  Enerji (kWh): 0.016224
  CO2 (kg): 0.007538
  Mod

{'Model': 'MobileNetV2',
 'Teknik': 'Pruning v2 (%50 Sparsity - Full Model)',
 'F1': 0.8405,
 'Precision': 0.8442,
 'Recall': 0.8406,
 'Accuracy': 0.8406,
 'Eğitim Süresi (sn)': 1584.48,
 'Enerji (kWh)': 0.016224,
 'CO2 (kg)': 0.007538,
 'Model Boyutu (MB)': 9.16}

In [15]:
# 1) Unfreezed Baseline - adil karşılaştırma için
base_model_uf = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model_uf.trainable = True

model_unfreezed = tf.keras.Sequential([
    base_model_uf,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation='softmax')
])

model_unfreezed.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop_uf = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

tracker_uf = OfflineEmissionsTracker(
    project_name="mobilenet_unfreezed",
    country_iso_code="TUR",
    log_level="error"
)

print("Unfreezed baseline başlıyor...")
tracker_uf.start()
start_time = time.time()

model_unfreezed.fit(
    x_train_final, y_train_final,
    validation_data=(x_val, y_val),
    epochs=5,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop_uf],
    verbose=1
)

train_time_uf = time.time() - start_time
tracker_uf.stop()

model_unfreezed.save('../results/mobilenet_unfreezed.keras')
uf_size = os.path.getsize('../results/mobilenet_unfreezed.keras') / (1024 * 1024)

y_pred_uf = np.argmax(model_unfreezed.predict(x_test, verbose=0), axis=1)

log_result(
    model_name='MobileNetV2',
    technique='Baseline (Unfreezed)',
    y_true=y_test,
    y_pred=y_pred_uf,
    train_time=train_time_uf,
    tracker_obj=tracker_uf,
    model_size_mb=uf_size
)

# 2) Sıkıştırılmış boyut karşılaştırması
import zipfile

def compressed_size(filepath):
    zip_path = filepath + '.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(filepath)
    size = os.path.getsize(zip_path) / (1024 * 1024)
    os.remove(zip_path)
    return size

baseline_compressed = compressed_size('../results/mobilenet_baseline.keras')
pruned_compressed = compressed_size('../results/mobilenet_pruned_v2.keras')
unfreezed_compressed = compressed_size('../results/mobilenet_unfreezed.keras')

print(f"\n--- Sıkıştırılmış Boyutlar ---")
print(f"Baseline (frozen):    {baseline_compressed:.2f} MB")
print(f"Baseline (unfreezed): {unfreezed_compressed:.2f} MB")
print(f"Pruned v2:            {pruned_compressed:.2f} MB")
print(f"Pruning sıkıştırma avantajı: {(1 - pruned_compressed/unfreezed_compressed)*100:.1f}%")

Unfreezed baseline başlıyor...
Epoch 1/5
1407/1407 [==============================] - 596s 382ms/step - loss: 1.0686 - accuracy: 0.6432 - val_loss: 1.6035 - val_accuracy: 0.5782
Epoch 2/5
1407/1407 [==============================] - 428s 304ms/step - loss: 0.5881 - accuracy: 0.7992 - val_loss: 0.5655 - val_accuracy: 0.8128
Epoch 3/5
1407/1407 [==============================] - 426s 303ms/step - loss: 0.4331 - accuracy: 0.8520 - val_loss: 0.4868 - val_accuracy: 0.8476
Epoch 4/5
1407/1407 [==============================] - 426s 302ms/step - loss: 0.3321 - accuracy: 0.8845 - val_loss: 0.4215 - val_accuracy: 0.8616
Epoch 5/5
1407/1407 [==============================] - 426s 303ms/step - loss: 0.2584 - accuracy: 0.9117 - val_loss: 0.4315 - val_accuracy: 0.8682

  Model: MobileNetV2
  Teknik: Baseline (Unfreezed)
  F1: 0.864
  Precision: 0.8664
  Recall: 0.8638
  Accuracy: 0.8638
  Eğitim Süresi (sn): 2398.31
  Enerji (kWh): 0.021275
  CO2 (kg): 0.009884
  Model Boyutu (MB): 26.34

--- Sıkış

In [16]:
# Güncellenmiş tabloyu kaydet
df_results = pd.DataFrame(results)
df_results.to_csv('../results/mobilenet_results.csv', index=False)
print(df_results.to_string(index=False))

      Model                                 Teknik     F1  Precision  Recall  Accuracy  Eğitim Süresi (sn)  Enerji (kWh)  CO2 (kg)  Model Boyutu (MB)
MobileNetV2              Baseline (Early Stopping) 0.6522     0.6550  0.6547    0.6547             1038.72      0.008409  0.003907               9.27
MobileNetV2                    Quantization (INT8) 0.6253     0.6336  0.6286    0.6286               19.93      0.000120  0.000056               2.60
MobileNetV2                 Pruning (%50 Sparsity) 0.6420     0.6434  0.6441    0.6441              377.24      0.002912  0.001353               9.16
MobileNetV2 Pruning v2 (%50 Sparsity - Full Model) 0.8405     0.8442  0.8406    0.8406             1584.48      0.016224  0.007538               9.16
MobileNetV2                   Baseline (Unfreezed) 0.8640     0.8664  0.8638    0.8638             2398.31      0.021275  0.009884              26.34
